# Resplit COOLANT HDF5 — Fix skewed train/dev/test ratio

Load the already-extracted `coolant_{train,dev,test}.h5` feature files, merge them back
together at the **article level** (so pairs from the same article never leak across
splits), then re-split into a new ratio and overwrite the HDF5 files in place.

Current split (from ViFactCheck's original files) is skewed: ~70/9/21 (train/dev/test
by article count). This notebook re-splits to the more standard **80/10/10**.

Originals are backed up to `backup_before_resplit/` before anything is overwritten.


In [ ]:
import os, sys, json, shutil
from collections import defaultdict
from pathlib import Path

import h5py
import numpy as np

# Resolve project root by walking upward until src/ found
PROJECT_ROOT = Path().resolve()
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / "src"))

# ── Config ────────────────────────────────────────────────────────────────
# Data lives on an external volume, independent of DATA_ROOT in .env.*
HDF5_DIR = Path("/Volumes/Private/Thesis_2026/thesis_data/processed_data/hdf5")

ORIGINAL_SPLITS = ["train", "dev", "test"]
NEW_SPLIT_RATIOS = {"train": 0.8, "dev": 0.1, "test": 0.1}
SEED = 42

live_files = {s: HDF5_DIR / f"coolant_{s}.h5" for s in ORIGINAL_SPLITS}
backup_dir = HDF5_DIR / "backup_before_resplit"
backup_files = {s: backup_dir / f"coolant_{s}.h5" for s in ORIGINAL_SPLITS}

# Always READ from the untouched backup if one exists, so re-running this notebook
# is idempotent (same seed + same original data = same split every time). Without
# this, a second run would resplit the already-resplit live files, silently producing
# a different partition even with the same SEED.
have_backup = all(p.exists() for p in backup_files.values())
source_files = backup_files if have_backup else live_files

print(f"HDF5_DIR: {HDF5_DIR}")
print(
    f"Reading from: {'backup_before_resplit/ (idempotent re-run)' if have_backup else 'live files (first run)'}"
)
for s, p in source_files.items():
    print(f"  {s:5}: {p} (exists={p.exists()})")

HDF5_DIR: /Volumes/Private/Thesis_2026/thesis_data/processed_data/hdf5
  train: coolant_train.h5 (exists=True)
  dev  : coolant_dev.h5 (exists=True)
  test : coolant_test.h5 (exists=True)


## Step 1: Load article-id metadata from each split

We only need `article_ids` (lightweight) to compute the new split assignment — the
heavy feature arrays are read later, only for the rows that end up in each new split.


In [2]:
meta = {}
for s in ORIGINAL_SPLITS:
    with h5py.File(source_files[s], "r") as f:
        article_ids = np.array(
            [a.decode() if isinstance(a, bytes) else a for a in f["article_ids"][:]]
        )
    meta[s] = article_ids
    print(
        f"{s:5}: {len(article_ids):5} rows, "
        f"{len(set(article_ids)):5} unique article_ids (local to this file)"
    )

train:  6724 rows,  3304 unique article_ids (local to this file)
dev  :   862 rows,   468 unique article_ids (local to this file)
test :  2053 rows,   967 unique article_ids (local to this file)


## Step 2: Group rows by article to avoid leakage

`article_ids` are only unique _within_ their own source file (e.g. `train` article `2`
and `dev` article `2` are unrelated articles), so the group key must include the source
split name.


In [3]:
groups = defaultdict(list)  # group_key -> [(src_split, row_idx), ...]
for s in ORIGINAL_SPLITS:
    for row_idx, aid in enumerate(meta[s]):
        groups[f"{s}__{aid}"].append((s, row_idx))

group_keys = list(groups.keys())
n_rows_total = sum(len(v) for v in groups.values())
print(f"Total rows (pairs): {n_rows_total}")
print(f"Total unique articles (groups): {len(group_keys)}")

Total rows (pairs): 9639
Total unique articles (groups): 4739


## Step 3: Shuffle & re-split at the article level


In [4]:
rng = np.random.RandomState(SEED)
shuffled = group_keys.copy()
rng.shuffle(shuffled)

n_total = len(shuffled)
n_train = int(round(n_total * NEW_SPLIT_RATIOS["train"]))
n_dev = int(round(n_total * NEW_SPLIT_RATIOS["dev"]))
n_test = n_total - n_train - n_dev  # remainder, avoids rounding loss

split_groups = {
    "train": shuffled[:n_train],
    "dev": shuffled[n_train : n_train + n_dev],
    "test": shuffled[n_train + n_dev :],
}

print("New split (article-level):")
for s, gk_list in split_groups.items():
    n_rows = sum(len(groups[k]) for k in gk_list)
    print(
        f"  {s:5}: {len(gk_list):5} articles ({len(gk_list)/n_total:5.1%})"
        f" -> {n_rows:5} pairs ({n_rows/n_rows_total:5.1%})"
    )

New split (article-level):
  train:  3791 articles (80.0%) ->  7645 pairs (79.3%)
  dev  :   474 articles (10.0%) ->  1020 pairs (10.6%)
  test :   474 articles (10.0%) ->   974 pairs (10.1%)


## Step 4: Map rows to their new split + a fresh (collision-free) article id

`CoolantPairDataset` calls `int(article_id)` and uses it to avoid sampling negatives
from the same article, so ids must stay integers and must be unique _within_ each new
split file. We assign a fresh sequential id per article group, scoped to its new split.


In [5]:
new_split_of_group = {}
new_id_of_group = {}
for ns, gk_list in split_groups.items():
    for new_id, gk in enumerate(gk_list):
        new_split_of_group[gk] = ns
        new_id_of_group[gk] = new_id

# new_split -> src_split -> [(row_idx, new_article_id), ...] sorted by row_idx
row_map = {ns: {s: [] for s in ORIGINAL_SPLITS} for ns in NEW_SPLIT_RATIOS}
for s in ORIGINAL_SPLITS:
    for row_idx, aid in enumerate(meta[s]):
        gk = f"{s}__{aid}"
        ns = new_split_of_group[gk]
        row_map[ns][s].append((row_idx, new_id_of_group[gk]))

for ns in row_map:
    for s in row_map[ns]:
        row_map[ns][s].sort(key=lambda t: t[0])

print("✓ Row mapping built.")

✓ Row mapping built.


## Step 5: Backup originals before touching anything


In [6]:
backup_dir.mkdir(parents=True, exist_ok=True)
for s, path in live_files.items():
    dest = backup_files[s]
    if dest.exists():
        print(f"  Backup already exists, skip: {dest.name}")
        continue
    shutil.copy2(path, dest)
    print(f"  Backed up {path.name} -> {dest}")

  Backed up coolant_train.h5 -> /Volumes/Private/Thesis_2026/thesis_data/processed_data/hdf5/backup_before_resplit/coolant_train.h5
  Backed up coolant_dev.h5 -> /Volumes/Private/Thesis_2026/thesis_data/processed_data/hdf5/backup_before_resplit/coolant_dev.h5
  Backed up coolant_test.h5 -> /Volumes/Private/Thesis_2026/thesis_data/processed_data/hdf5/backup_before_resplit/coolant_test.h5


## Step 6: Extract new split arrays and write to temp files

Written to `coolant_{split}.h5.tmp` first (sources are read from the originals, which
are untouched until the final swap in Step 7).


In [7]:
def extract_new_split(new_split_name):
    caption_list, image_list = [], []
    article_ids_list, source_labels_list = [], []
    source_urls_list, image_paths_list, folder_paths_list = [], [], []

    for src_split in ORIGINAL_SPLITS:
        pairs = row_map[new_split_name][src_split]
        if not pairs:
            continue
        idx = np.array([p[0] for p in pairs], dtype=np.int64)
        new_ids = np.array([p[1] for p in pairs], dtype=np.int64)

        with h5py.File(source_files[src_split], "r") as f:
            caption_list.append(f["caption_features"][idx])
            image_list.append(f["image_features"][idx])
            source_labels_list.append(f["source_labels"][idx])
            source_urls_list.append(f["source_urls"][idx])
            image_paths_list.append(f["image_paths"][idx])
            folder_paths_list.append(f["folder_paths"][idx])
        article_ids_list.append(np.array([str(i) for i in new_ids], dtype="S"))

    return {
        "caption_features": np.concatenate(caption_list, axis=0),
        "image_features": np.concatenate(image_list, axis=0),
        "article_ids": np.concatenate(article_ids_list, axis=0),
        "source_labels": np.concatenate(source_labels_list, axis=0),
        "source_urls": np.concatenate(source_urls_list, axis=0),
        "image_paths": np.concatenate(image_paths_list, axis=0),
        "folder_paths": np.concatenate(folder_paths_list, axis=0),
    }


tmp_files = {}
for new_split_name in NEW_SPLIT_RATIOS:
    print(f"\n{new_split_name.upper()}: extracting...")
    data = extract_new_split(new_split_name)

    with h5py.File(source_files["train"], "r") as f_ref:
        base_attrs = dict(f_ref.attrs)

    tmp_path = HDF5_DIR / f"coolant_{new_split_name}.h5.tmp"
    with h5py.File(tmp_path, "w") as f:
        for name, arr in data.items():
            f.create_dataset(name, data=arr)

        f.attrs["n_samples"] = len(data["article_ids"])
        f.attrs["caption_shape"] = data["caption_features"].shape
        f.attrs["image_shape"] = data["image_features"].shape
        f.attrs["text_model"] = base_attrs.get("text_model", "")
        f.attrs["image_model"] = base_attrs.get("image_model", "")
        f.attrs["max_length"] = base_attrs.get("max_length", 0)
        f.attrs["data_source"] = base_attrs.get("data_source", "")
        f.attrs["label_variant"] = base_attrs.get("label_variant", "")
        f.attrs["resplit_ratio"] = str(NEW_SPLIT_RATIOS)
        f.attrs["resplit_seed"] = SEED

    tmp_files[new_split_name] = tmp_path
    print(
        f"  Wrote {tmp_path.name}: "
        f"caption {data['caption_features'].shape}, image {data['image_features'].shape}"
    )


TRAIN: extracting...
  Wrote coolant_train.h5.tmp: caption (7645, 128, 768), image (7645, 2048)

DEV: extracting...
  Wrote coolant_dev.h5.tmp: caption (1020, 128, 768), image (1020, 2048)

TEST: extracting...
  Wrote coolant_test.h5.tmp: caption (974, 128, 768), image (974, 2048)


## Step 7: Swap temp files into place


In [8]:
for s, tmp_path in tmp_files.items():
    final_path = live_files[s]
    os.replace(tmp_path, final_path)
    print(f"  {tmp_path.name} -> {final_path.name}")

print("\n✓ Resplit complete. Originals preserved in backup_before_resplit/.")

  coolant_train.h5.tmp -> coolant_train.h5
  coolant_dev.h5.tmp -> coolant_dev.h5
  coolant_test.h5.tmp -> coolant_test.h5

✓ Resplit complete. Originals preserved in backup_before_resplit/.


## Step 8: Verify — no article leakage, correct row counts


In [9]:
print("Verifying resplit HDF5 files...\n")

for s in ORIGINAL_SPLITS:
    with h5py.File(live_files[s], "r") as f:
        n_samples = f.attrs["n_samples"]
        caption_shape = f["caption_features"].shape
        image_shape = f["image_features"].shape
        n_articles = len(set(f["article_ids"][:]))
        print(f"{s}:")
        print(f"  Samples : {n_samples}")
        print(f"  Articles: {n_articles}")
        print(f"  Caption : {caption_shape}")
        print(f"  Image   : {image_shape}")

print(
    "\nSanity check: no article group appears in more than one new split (by construction)."
)
assert set(split_groups["train"]).isdisjoint(split_groups["dev"])
assert set(split_groups["train"]).isdisjoint(split_groups["test"])
assert set(split_groups["dev"]).isdisjoint(split_groups["test"])
print("✓ No leakage between splits.")

stats_file = HDF5_DIR / "preprocessing_stats.json"
report_data = []
for s in ORIGINAL_SPLITS:
    with h5py.File(live_files[s], "r") as f:
        report_data.append(
            {
                "split": s,
                "raw_articles": len(set(f["article_ids"][:])),
                "valid_pairs": int(f.attrs["n_samples"]),
                "hdf5_rows": int(f.attrs["n_samples"]),
                "caption_shape": str(f.attrs["caption_shape"]),
                "image_shape": str(f.attrs["image_shape"]),
                "file_size_mb": f"{live_files[s].stat().st_size / (1024 * 1024):.2f}",
                "resplit_ratio": str(NEW_SPLIT_RATIOS),
                "resplit_seed": SEED,
            }
        )
with open(stats_file, "w", encoding="utf-8") as f:
    json.dump(report_data, f, ensure_ascii=False, indent=2)
print(f"\n✓ Stats saved to {stats_file.name}")

Verifying resplit HDF5 files...

train:
  Samples : 7645
  Articles: 3791
  Caption : (7645, 128, 768)
  Image   : (7645, 2048)
dev:
  Samples : 1020
  Articles: 474
  Caption : (1020, 128, 768)
  Image   : (1020, 2048)
test:
  Samples : 974
  Articles: 474
  Caption : (974, 128, 768)
  Image   : (974, 2048)

Sanity check: no article group appears in more than one new split (by construction).
✓ No leakage between splits.

✓ Stats saved to preprocessing_stats.json
